# GameTheory-04d-Marchandage-Asymetrique-CSharp

**Durée** : 75 minutes.

**Navigation** : [Version Python](GameTheory-04d-Marchandage-Asymetrique-Python.ipynb) | [<< 4-NashEquilibrium](GameTheory-04-NashEquilibrium-Python.ipynb) | [Index](README.md) | [4e-Reflective-Oracles >>](GameTheory-04e-Reflective-Oracles-Python.ipynb)

## Marchandage asymétrique : dépendance, désir et point de désaccord

### Objectifs d'apprentissage

1. Formuler le point de désaccord (Nash, 1950) comme réponse à « que se passe-t-il si rien ne s'accorde ? ».
2. Distinguer le **désir** (une composante du faisceau) de la **dépendance** (le faisceau entier).
3. Construire un contre-exemple au principe du moindre intérêt et en tester la robustesse au générateur de poids.
4. Lire honnêtement la dissociation relationnelle : l'émancipation externe réduit le levier brut mais concentre l'asymétrie relationnelle.

### Prérequis
Équilibres de Nash purs et mixtes ([GT-04](GameTheory-04-NashEquilibrium-Python.ipynb)) ; intuitions de base
sur les préférences et les alternatives extérieures (`CLalt`).


In [1]:
using System;
using System.Linq;
using System.Collections.Generic;
using System.Globalization;
CultureInfo.DefaultThreadCurrentCulture=CultureInfo.InvariantCulture;
Console.WriteLine($"Runtime : {System.Runtime.InteropServices.RuntimeInformation.FrameworkDescription} ; calcul BCL .NET");

Runtime : .NET 10.0.14 ; calcul BCL .NET


## Le point de désaccord et le pouvoir : dépendance, désir et point de désaccord

L'équilibre de Nash dit quand plus rien ne bouge ; le **point de désaccord**
du marchandage (Nash, 1950) dit ce que chacun obtient si rien ne s'accorde.
Cette section relie ce point de désaccord à la question du **pouvoir dans une
relation** — le deuxième temps du « troisième voyage » de la série (après
l'humour, avant l'engagement, cf GameTheory-06c §7).

L'énoncé populaire — « celui qui désire le moins contrôle le plus » — est une
hypothèse simplifiée que ce modèle met à l’épreuve ; le résultat calculé ne suffit pas
à juger une loi psychologique ou sociologique empirique. Le tableau suivant conserve
les repères historiques du source Python, comme contexte :

| Notion | Nom canonique | Ce qu'elle dit exactement |
|--------|---------------|---------------------------|
| L'asymétrie de dépendance | **principe du moindre intérêt** (Waller, 1938) | celui dont l'intérêt au maintien de la relation est le plus faible dispose du levier |
| Sa forme relationnelle | **power-dependence** (Emerson, 1962) | le pouvoir de A sur B est la **dépendance de B envers A** — pas un trait de A |
| Sa forme psychologique | théorie de l'**interdépendance** (Thibaut & Kelley, 1959) | `CLalt` = qualité de la meilleure alternative disponible |
| Sa forme économique | **point de désaccord** (Nash, 1950) ; offres alternées (Rubinstein, 1982) | le partage dépend de ce que chacun obtient si l'accord échoue |
| Son maintien dans le temps | **modèle d'investissement** (Rusbult, 1980) | engagement ≈ satisfaction + investissements − alternatives |

Le gain de ce vocabulaire : il rend l'énoncé **faux de la bonne manière**. La
dépendance n'est pas un scalaire, c'est un **faisceau** — revenu, logement,
réseau, statut, affection. Le désir n'en est **qu'une composante**. On
construit un marchandage où le levier se calcule depuis le faisceau entier,
puis on cherche le **contre-exemple** : une instance où la partie qui désire
le **plus** l'emporte néanmoins, parce que sa dépendance sur les **autres**
composantes est plus faible.

In [2]:
string[] composantes={"affection","revenu","logement","reseau","statut"};
(Dictionary<string,double> gaps,double dep) Faisceau(Dictionary<string,double> v,Dictionary<string,double> o,Dictionary<string,double> w){var g=composantes.ToDictionary(k=>k,k=>w[k]*Math.Max(v[k]-o[k],0));return(g,g.Values.Sum());}
(double partA,double partB) Partage(double da,double db,double kappa=6,bool constants=false){if(constants)return(.5,.5);double xi=1/(1+Math.Exp(-kappa*(da-db)));return(1-xi,xi);}
var vA=new Dictionary<string,double>{{"affection",.9},{"revenu",.8},{"logement",.7},{"reseau",.6},{"statut",.6}};
var oA=new Dictionary<string,double>{{"affection",.1},{"revenu",.8},{"logement",.7},{"reseau",.6},{"statut",.6}};
var vB=new Dictionary<string,double>{{"affection",.6},{"revenu",.7},{"logement",.7},{"reseau",.5},{"statut",.5}};
var oB=new Dictionary<string,double>{{"affection",.3},{"revenu",.2},{"logement",.1},{"reseau",.5},{"statut",.2}};
var wA=composantes.ToDictionary(k=>k,k=>.2);var wB=composantes.ToDictionary(k=>k,k=>.2);
var (gapsA,da)=Faisceau(vA,oA,wA);var (gapsB,db)=Faisceau(vB,oB,wB);double desirA=gapsA["affection"],desirB=gapsB["affection"];
foreach(var k in composantes)Console.WriteLine($"{k,-10} A={gapsA[k]:F3}, B={gapsB[k]:F3}");Console.WriteLine($"D_A={da:F3}, D_B={db:F3}; désir A={desirA:F3}, B={desirB:F3}");

affection  A=0.160, B=0.060


revenu     A=0.000, B=0.100


logement   A=0.000, B=0.120


reseau     A=0.000, B=0.000


statut     A=0.000, B=0.060


D_A=0.160, D_B=0.340; désir A=0.160, B=0.060


### Lecture du faisceau

La somme des écarts donne une dépendance de 0,160 pour A et 0,340 pour B. Le désir affectif seul ne résume donc pas le faisceau.

In [3]:
var (partA,xiB)=Partage(da,db);
Console.WriteLine($"Désir A={desirA:F3} > B={desirB:F3} : {desirA>desirB}; part de A={partA:F3}, B={xiB:F3}; contre-exemple du modèle={desirA>desirB&&partA>.5}");

Désir A=0.160 > B=0.060 : True; part de A=0.746, B=0.254; contre-exemple du modèle=True


**Statut explicite de `Partage` (`partage_nash` côté Python).** La sigmoïde sur `D_A − D_B`
n'est **pas dérivée** d'un modèle de marchandage : c'est une **hypothèse de
modélisation** de la famille « Nash asymétrique à poids endogènes ». `kappa`
est un paramètre libre fixé à 6,0. Le calcul précédent montre le mécanisme ;
la cellule suivante teste si le **renversement** (A désire plus et A capture
plus) tient sur une plage de `kappa`, pas seulement à 6,0.

In [4]:
foreach(double k in new[]{0.0,1,2,4,6,8,12,20}){var (part,_)=Partage(da,db,k);Console.WriteLine($"kappa={k:F1}, part A={part:F3}, D_B>D_A={db>da}, renversement={desirA>desirB&&part>.5}");}
Console.WriteLine($"D_B-D_A={db-da:F3} ; la direction pour kappa positif dépend du signe, l'intensité dépend de kappa.");

kappa=0.0, part A=0.500, D_B>D_A=True, renversement=False


kappa=1.0, part A=0.545, D_B>D_A=True, renversement=True


kappa=2.0, part A=0.589, D_B>D_A=True, renversement=True


kappa=4.0, part A=0.673, D_B>D_A=True, renversement=True


kappa=6.0, part A=0.746, D_B>D_A=True, renversement=True


kappa=8.0, part A=0.808, D_B>D_A=True, renversement=True


kappa=12.0, part A=0.897, D_B>D_A=True, renversement=True


kappa=20.0, part A=0.973, D_B>D_A=True, renversement=True


D_B-D_A=0.180 ; la direction pour kappa positif dépend du signe, l'intensité dépend de kappa.


**Lecture du contre-exemple.** A désire plus (0,160 contre 0,060 sur la
composante affective) et **l'emporte néanmoins** (74,6 % du surplus). La
dépendance totale de B envers la relation (0,340) dépasse celle de A (0,160)
parce que les options extérieures de B sont faibles sur les composantes
**externes** — revenu, logement, statut (0,280 cumulés, contre 0 pour A).

Dans ce modèle stylisé, l'énoncé qui assimile intérêt et seul désir n'est pas une règle universelle : dire « le
désir commande » est faux dès que le faisceau contient autre chose que le
désir. Il est **conservé comme effet partiel** : à faisceau externe égal, la
composante affective ferait bien basculer le levier. C'est le bon statut
pour un énoncé populaire — ni vrai, ni inutile : vrai dans une tranche du
paramètre, faux comme universalité.

In [5]:
(double,double) PartageExogene(double za,double zb)=>(za/(za+zb),zb/(za+zb));
double zaExo=oA.Values.Average(),zbExo=oB.Values.Average();double partEndo=Partage(da,db).partA,partExo=PartageExogene(zaExo,zbExo).Item1,partConst=Partage(da,db,constants:true).partA;
Console.WriteLine($"Parts A : endogène={partEndo:F3}, exogène={partExo:F3}, constant={partConst:F3}; z_A={zaExo:F2}, z_B={zbExo:F2}");
var rng=new Random(13313);double Normal(){double u1=Math.Max(rng.NextDouble(),double.Epsilon),u2=rng.NextDouble();return Math.Sqrt(-2*Math.Log(u1))*Math.Cos(2*Math.PI*u2);}
Dictionary<string,double> Perturbe(Dictionary<string,double> x)=>composantes.ToDictionary(k=>k,k=>Math.Clamp(x[k]+.08*Normal(),0,1));
int n=200,nPre=0;var repro=new Dictionary<string,int>{{"endogene",0},{"exogene",0},{"constant",0}};
for(int i=0;i<n;i++){var vap=Perturbe(vA);var oap=Perturbe(oA);var vbp=Perturbe(vB);var obp=Perturbe(oB);var (gapA,dap)=Faisceau(vap,oap,wA);var (gapB,dbp)=Faisceau(vbp,obp,wB);if(gapA["affection"]<=gapB["affection"])continue;nPre++;if(Partage(dap,dbp).partA>.5)repro["endogene"]++;if(PartageExogene(oap.Values.Average(),obp.Values.Average()).Item1>.5)repro["exogene"]++;if(Partage(dap,dbp,constants:true).partA>.5)repro["constant"]++;}
Console.WriteLine($"{n} instances perturbées ; précondition {nPre} fois");foreach(var x in repro)Console.WriteLine($"{x.Key} : {x.Value}/{nPre} ({100.0*x.Value/Math.Max(nPre,1):F0} %)");

Parts A : endogène=0.746, exogène=0.683, constant=0.500; z_A=0.56, z_B=0.26


200 instances perturbées ; précondition 200 fois


endogene : 199/200 (100 %)


exogene : 200/200 (100 %)


constant : 0/200 (0 %)


### Lecture des générateurs

Les comptes des 200 perturbations comparent les deux familles informatives à un poids constant. Les taux dépendent des tirages et des hypothèses choisies.

In [6]:
(double,double) PartageDehors(Dictionary<string,double> oa,Dictionary<string,double> ob,double gamma=2)=>PartageExogene(Math.Pow(oa.Values.Average(),gamma),Math.Pow(ob.Values.Average(),gamma));
double pDehors1=PartageDehors(oA,oB,1).Item1;Console.WriteLine($"gamma=1 : couche={pDehors1:F3}, substrat={partExo:F3}, équivalence={Math.Abs(pDehors1-partExo)<1e-12}");

gamma=1 : couche=0.683, substrat=0.683, équivalence=True


### Lecture de la couche extérieure

Pour gamma égal à 1, le calcul reproduit le ratio exogène de référence.

In [7]:
foreach(double aff in new[]{.5,.7,.9,1.0}){var v=new Dictionary<string,double>(vA);v["affection"]=aff;var (g,d)=Faisceau(v,oA,wA);Console.WriteLine($"affection={aff:F1}, désir={g["affection"]:F3}, endogène={Partage(d,db).partA:F3}, exogène={PartageDehors(oA,oB).Item1:F3}");}

affection=0.5, désir=0.080, endogène=0.826, exogène=0.823


affection=0.7, désir=0.120, endogène=0.789, exogène=0.823


affection=0.9, désir=0.160, endogène=0.746, exogène=0.823


affection=1.0, désir=0.180, endogène=0.723, exogène=0.823


**Lecture chiffree — le canal du desir appartient a la famille, pas au dispositif.** Le tableau fait monter la valeur affective de A : son desir rendu passe de `0.080` a `0.180` (plus que le double). Cote endogene, le levier de A recule de `0.826` a `0.723` (perte 0.103 sur les quatre lignes) -- plus A desire, moins il capte : le moindre interet intra-famille. Cote exogene, `0.823` constant : le poids ne consulte que les options exterieures, ce mouvement n'existe pas pour lui. Les deux familles partent presque confondues (a v_affection = 0.5, `0.826` contre `0.823`, ecart 0.003) et finissent separees de 0.100 (0.823 - 0.723 a v_affection = 1.0) : changer de generateur n'est pas changer un reglage, c'est ouvrir ou fermer un canal.


### Exercice 1 — Faisceaux extérieurs égaux

Complétez `exterieursEgaux` pour produire deux copies indépendantes du faisceau fourni, puis `compareFamilles` pour retourner les parts de A dans les familles endogène et exogène. Comparez le dehors de B et un faisceau neutre à 0,3 sur chaque composante ; expliquez quelle famille perd l'information quand les dehors sont égaux.

In [8]:
Func<Dictionary<string,double>,(Dictionary<string,double>,Dictionary<string,double>)?> exterieursEgaux=commun=>{
 // TODO : deux copies indépendantes
 return null;
};
Func<Dictionary<string,double>,Dictionary<string,double>,(double,double)?> compareFamilles=(oa,ob)=>{
 // TODO : parts endogène et exogène de A
 return null;
};
var casDehors=new[]{new Dictionary<string,double>(oB),composantes.ToDictionary(k=>k,k=>.3)};var avantDehors=casDehors.Select(x=>new Dictionary<string,double>(x)).ToArray();var comparaisonTemoin=compareFamilles(new Dictionary<string,double>(oA),new Dictionary<string,double>(oB));var repDehors=casDehors.Select(exterieursEgaux).ToArray();if(casDehors.Where((x,i)=>x.Count!=avantDehors[i].Count||composantes.Any(k=>!x.ContainsKey(k)||x[k]!=avantDehors[i][k])).Any())throw new Exception("Entrées modifiées");
if(repDehors.All(x=>x is null)&&comparaisonTemoin is null)Console.WriteLine("Exercice 1 à compléter");
else{if(repDehors.Any(x=>x is null)||comparaisonTemoin is null)throw new Exception("Réponse partielle");for(int i=0;i<repDehors.Length;i++){var x=repDehors[i].Value;if(x.Item1==null||x.Item2==null||object.ReferenceEquals(x.Item1,x.Item2)||object.ReferenceEquals(x.Item1,casDehors[i])||object.ReferenceEquals(x.Item2,casDehors[i])||x.Item1.Count!=5||x.Item2.Count!=5||composantes.Any(k=>!x.Item1.ContainsKey(k)||!x.Item2.ContainsKey(k)||x.Item1[k]!=casDehors[i][k]||x.Item2[k]!=casDehors[i][k]))throw new Exception("Copies incorrectes");}
 var casParts=repDehors.Select(x=>x.Value).Concat(new[]{(new Dictionary<string,double>(oA),new Dictionary<string,double>(oB))}).ToArray();var reps=casParts.Select(x=>compareFamilles(x.Item1,x.Item2)).ToArray();
 if(reps.Any(x=>x is null))throw new Exception("Réponse partielle");else{for(int i=0;i<reps.Length;i++){var x=reps[i].Value;var (oa,ob)=casParts[i];double attendu=Partage(Faisceau(vA,oa,wA).dep,Faisceau(vB,ob,wB).dep).partA,exo=PartageExogene(oa.Values.Average(),ob.Values.Average()).Item1;if(!double.IsFinite(x.Item1)||!double.IsFinite(x.Item2)||Math.Abs(x.Item1-attendu)>1e-9||Math.Abs(x.Item2-exo)>1e-9)throw new Exception("Parts incorrectes");}Console.WriteLine("Exercice 1 : sondes validées");}}

Exercice 1 à compléter


### Transition

Après avoir comparé des dehors égaux, on examine la sensibilité à la pente gamma sur les faisceaux initiaux.

In [9]:
foreach(double g in new[]{.5,1,2,4,6,8}){double part=PartageDehors(oA,oB,g).Item1;Console.WriteLine($"gamma={g:F1}, part A exogène={part:F3}, contre-exemple={part>.5}");}Console.WriteLine($"moyennes dehors : A={oA.Values.Average():F3}, B={oB.Values.Average():F3}");

gamma=0.5, part A exogène=0.595, contre-exemple=True


gamma=1.0, part A exogène=0.683, contre-exemple=True


gamma=2.0, part A exogène=0.823, contre-exemple=True


gamma=4.0, part A exogène=0.956, contre-exemple=True


gamma=6.0, part A exogène=0.990, contre-exemple=True


gamma=8.0, part A exogène=0.998, contre-exemple=True


moyennes dehors : A=0.560, B=0.260


**Verdict de robustesse — comparaisons dans les générateurs étudiés.** Une objection méthodologique se pose : si le poids de négociation est lui-même une fonction de l'écart de dépendance, le contre-exemple à la lecture du moindre intérêt par le seul désir peut être tautologique — posée par le modélisateur plutôt que mesurée. L'interrogation à trois générateurs tranche : **la direction observée se retrouve dans les deux familles informatives étudiées**.

| Générateur | Le poids dépend de | `part_A` | Contre-exemple |
|------|--------------------|----------|----------------|
| Endogène (κ=6) | écart `D_A − D_B` (construit du modèle) | 0,746 | OUI (199/200) |
| Exogène γ=1 (clout brut) | capital extérieur `ō` (options dehors) | 0,683 | OUI (200/200) |
| Exogène γ=2 (couche `partage_nash_dehors`) | capital extérieur `ō`, pente γ | 0,823 | OUI |
| Constants (contrôle) | rien — canal coupé | 0,500 | non (neutralité) |

1. **Le modèle ne confirme pas la prédiction directionnelle dans ces trois bras.** La version simplifiée qui assimile l'intérêt au seul désir prédit `part_A < 0,5` quand `desi_A > desi_B`. Sur l’instance de départ, aucun des trois bras ne donne cette prédiction. Même le contrôle constants — qui coupe **tout** canal vers le poids — donne 0,500, pas la prédiction directionnelle du principe : sans information sur le faisceau, le désir seul ne prédit rien. Cette expérience numérique ne constitue pas une réfutation empirique générale : elle teste des hypothèses de poids posées dans le modèle.
2. **Le renversement positif (« A désire plus et l'emporte ») appartient au dispositif (2/2 familles informatives : 199/200 et 200/200).** Deux chemins structurellement distincts — l'écart de dépendance et le capital extérieur — donnent la même direction sur cette instance. Le différenciateur ci-dessus montre que les deux familles ne réagissent pas aux mêmes variables ; cela ne dispense pas de leurs hypothèses postulées.
3. **Borne 1 — l'intensité est propre à la famille et à sa pente.** L'écart à la ligne de base va de +0,183 (exogène γ=1) à +0,246 (endogène κ=6) et +0,323 (γ=2) ; le balayage γ ∈ [0,5 ; 8] donne 0,595 → 0,998, exactement comme le levier endogène s'élargissait avec κ. La **direction** est une propriété du dispositif, le **chiffre** une propriété de la fonction.
4. **Borne 2 — l'effet partiel est conditionnel à la famille endogène.** « À faisceau externe égal, le désir fait basculer le levier » n'existe que dans la famille où le poids voit le faisceau ; dans la famille exogène, le désir n'a aucun canal (différenciateur : 0,823 constant). La lecture précédente reste exacte **pour le modèle posé** ; généraliser l'effet partiel à toute famille de poids serait un artefact.

**Statut :** contre-exemple dans ces familles de poids postulées ; le chiffre 74,6 % et l'effet partiel sont des propriétés de la famille endogène, à lire comme telles.

In [10]:
foreach(double t in new[]{0.0,.2,.4,.6,.8,1.0}){var ob=new Dictionary<string,double>(oB);foreach(var k in new[]{"revenu","logement","statut"})ob[k]=Math.Min(oB[k]+t*(vB[k]-oB[k]),vB[k]);var (g,d)=Faisceau(vB,ob,wB);double part=Partage(da,d).partA,aff=d>1e-9?g["affection"]/d:double.NaN;Console.WriteLine($"étape={t:F1}, D_B={d:F3}, part_A={part:F3}, affection/D_B={aff:P1}, affection/D_A={gapsA["affection"]/da:P1}");}

étape=0.0, D_B=0.340, part_A=0.746, affection/D_B=17.6 %, affection/D_A=100.0 %


étape=0.2, D_B=0.284, part_A=0.678, affection/D_B=21.1 %, affection/D_A=100.0 %


étape=0.4, D_B=0.228, part_A=0.601, affection/D_B=26.3 %, affection/D_A=100.0 %


étape=0.6, D_B=0.172, part_A=0.518, affection/D_B=34.9 %, affection/D_A=100.0 %


étape=0.8, D_B=0.116, part_A=0.434, affection/D_B=51.7 %, affection/D_A=100.0 %


étape=1.0, D_B=0.060, part_A=0.354, affection/D_B=100.0 %, affection/D_A=100.0 %


**Lecture de la bascule — la dissociation historique.** Émanciper des
dépendances **externes** (revenu, statut, autonomie juridique et résidentielle)
est un progrès ; l'effet mécanique dans ce modèle est double : le levier brut
de l'autre partie **décline** (0,746 → 0,354) — et la part **relative** de
l'asymétrie relationnelle restante **monte** (18 % → 100 % de la dépendance de
B). L'asymétrie restante ne devient pas plus grande en absolu : elle devient
presque **toute** l'asymétrie visible. C'est la formulation exacte de la
dissociation :

```
émancipation des dépendances externes  ≠>  disparition des asymétries relationnelles
```

Le modèle ne dit pas qu'il faudrait regretter la contrainte ancienne — il dit
que la négociation change de **nature** : moins d'enjeux de survie, plus
d'enjeux relationnels purs. Rien dans ce résultat ne peut servir d'argument
contre l'émancipation.

### Exercice 2 — Émancipation composante par composante

Complétez `emancipationUnitaire` et `emancipationJointe` : retournez dépendance de B, part de A et part de l'affection pour chaque composante, puis pour revenu, logement et statut ensemble. Comparez les déplacements à la référence. L'intervention sur l'affection est un contre-factuel mathématique, pas une hypothèse d'émancipation extérieure réelle.

In [11]:
Func<string,(double,double,double)?> emancipationUnitaire=k=>{
 // TODO : dépendance, part de A et part de l'affection
 return null;
};
Func<(double,double,double)?> emancipationJointe=()=>{
 // TODO : revenu, logement et statut ensemble
 return null;
};
var etatInitial=new Dictionary<string,double>(oB);var repEmancipation=new (double,double,double)?[composantes.Length];
for(int i=0;i<composantes.Length;i++){repEmancipation[i]=emancipationUnitaire(composantes[i]);if(composantes.Any(k=>oB[k]!=etatInitial[k]))throw new Exception("Options modifiées");}
var repJointe=emancipationJointe();if(composantes.Any(k=>oB[k]!=etatInitial[k]))throw new Exception("Options modifiées");
if(repEmancipation.All(x=>x is null)&&repJointe is null)Console.WriteLine("Exercice 2 à compléter");
else{if(repEmancipation.Any(x=>x is null)||repJointe is null)throw new Exception("Réponse partielle");var tout=repEmancipation.Select(x=>x.Value).Concat(new[]{repJointe.Value}).ToArray();var groupes=composantes.Select(k=>new[]{k}).Concat(new[]{new[]{"revenu","logement","statut"}}).ToArray();for(int i=0;i<tout.Length;i++){var ob=new Dictionary<string,double>(oB);foreach(string k in groupes[i])ob[k]=vB[k];var(g,d)=Faisceau(vB,ob,wB);var x=tout[i];double a=Partage(da,d).partA,aff=d>1e-9?g["affection"]/d:0;if(!double.IsFinite(x.Item1)||!double.IsFinite(x.Item2)||!double.IsFinite(x.Item3)||Math.Abs(x.Item1-d)>1e-9||Math.Abs(x.Item2-a)>1e-9||Math.Abs(x.Item3-aff)>1e-9)throw new Exception("Sonde incorrecte");}try{oB["revenu"]+=.07;var modifie=new Dictionary<string,double>(oB);var test=emancipationUnitaire("logement");if(test is null||composantes.Any(k=>oB[k]!=modifie[k]))throw new Exception("État modifié");var ob=new Dictionary<string,double>(oB);ob["logement"]=vB["logement"];var (g,d)=Faisceau(vB,ob,wB);var x=test.Value;if(!double.IsFinite(x.Item1)||!double.IsFinite(x.Item2)||!double.IsFinite(x.Item3)||Math.Abs(x.Item1-d)>1e-9||Math.Abs(x.Item2-Partage(da,d).partA)>1e-9||Math.Abs(x.Item3-(d>1e-9?g["affection"]/d:0))>1e-9)throw new Exception("Sonde perturbée incorrecte");}finally{foreach(var k in composantes)oB[k]=etatInitial[k];} Console.WriteLine("Exercice 2 : sondes validées ; déplacements : "+string.Join(", ",tout.Select(x=>x.Item2-partA)));}

Exercice 2 à compléter


**Deux questions ouvertes — la part qui pose des questions, pas des variables**

La psychanalyse et la sociologie compréhensive entrent ici comme
**productrices de distinctions**, jamais de paramètres. Une question écrite,
suivie de ce qu'il faudrait mesurer pour y répondre — et d'aucun coefficient.

> **1. Désir mimétique (Girard, 1961).** La valeur d'un objet dépend du fait
> qu'un tiers le désire. Le modèle ci-dessus traite `o_k` et `v_k` comme
> exogènes. *Que faudrait-il ajouter pour que la valeur de la relation devienne
> endogène au désir d'autrui — et qu'est-ce qui, dans une trajectoire observée,
> permettrait de distinguer un désir de l'objet d'un désir par l'intermédiaire
> de l'objet ?*
>
> Ce qu'il faudrait mesurer : une variation du « désirabilité » de la relation
> **indépendante de ses propriétés intrinsèques** — par exemple une variation
> de sa visibilité sociale auprès de tiers — et l'effet de cette variation sur
> `v_affection` estimé avant/après. Sans cette exogénéité contrôlée, mimétisme
> et préférence stable sont indiscernables.

> **2. Demande versus désir (Lacan) / espace transitionnel (Winnicott).** La
> demande est formulable et négociable ; le désir reste au-delà. Le modèle
> réduit tout le faisceau à des grandeurs négociables. *Quelle observabilité
> faudrait-il pour distinguer, dans le comportement de négociation d'un agent,
> ce qui relève de la demande satisfaite de ce qui relève du désir maintenu —
> sachant qu'un agent peut refuser un accord avantageux précément pour
> préserver le désir ?*
>
> Ce qu'il faudrait mesurer : des **refus d'accords dominants** — des
> trajectoires où l'agent décline une part de surplus supérieure à son point de
> désaccord. Dans le modèle, ce comportement est une erreur ; hors du modèle,
> c'est un signal. La fréquence et la sélectivité de ces refus (sur quelles
> composantes ?) seraient l'observable.

La forme interdite, pour mémoire : `coefficient_mimetique = 0.3` — un
paramètre qui prétend mesurer ce qu'il vient d'illustrer ne mesure rien.

**Limites et anti-claims.** Ce marchandage est un **modèle partiel** du
désir, explicitement — jamais sa réduction. La négociation est ici un des
régimes possibles de la relation, pas son essence ; l'utilité n'est pas le
désir ; l'amour n'est pas une symétrie de dépendances. Aucune inférence d'état
mental attribué à une personne réelle : on étudie des **structures
d'incitations**, pas des personnes. Les composantes (affection, revenu,
logement, réseau, statut) sont des dimensions d'analyse sociologique, pas un
inventaire de la vie relationnelle  ; ils servent seulement aux calculs conditionnels affichés ici.

**Notices historiques conservées du source** : Waller, W. (1938), *The Family: A
Dynamic Interpretation* ; Emerson, R. M. (1962), « Power-Dependence
Relations », *American Sociological Review* 27(1) ; Thibaut, J. W. & Kelley,
H. H. (1959), *The Social Psychology of Groups* ; Nash, J. F. (1950), « The
Bargaining Problem », *Econometrica* 18(2) ; Rubinstein, A. (1982), « Perfect
Equilibrium in a Bargaining Model », *Econometrica* 50(1) ; Rusbult, C. E.
(1980), « Commitment and satisfaction in romantic associations », *Journal of
Experimental Social Psychology* 16(2) ; Girard, R. (1961), *Mensonge
romantique et vérité romanesque*.

La comparaison des générateurs porte sur cette instance et ces perturbations simulées ; elle ne démontre pas une loi empirique générale.


**Référence pédagogique primaire** : [source Python versionné](https://github.com/jsboige/CoursIA/blob/a8566836bb4246cfedbf6923f63fb0b2ab37d368/MyIA.AI.Notebooks/GameTheory/GameTheory-04d-Marchandage-Asymetrique-Python.ipynb). Les notices et distinctions théoriques de ce carnet proviennent de ce source ; aucune nouvelle lecture de copie archivée ni vérification de publication de première main n’est revendiquée par ce jumeau. Les sorties mesurent un modèle stylisé, pas des données empiriques sur des personnes.

### Exercice 3 — Valeur affective et levier

Complétez `exerciceLevierAffection` : conservez les entrées et calculez la part endogène de A pour chacune. Vérifiez la variation et interprétez-la à l'aide du tableau précédent.

In [12]:
record Levier(double[] Affection,double[] PartA);
Func<double[],Levier> exerciceLevierAffection=valeurs=>{
 // TODO : conserver les entrées et calculer les parts
 return null;
};
var casAffection=new[]{new[]{.5,.7,.9},new[]{.25,.8,1.0},Array.Empty<double>()};var repAffection=casAffection.Select(exerciceLevierAffection).ToArray();
if(repAffection.All(x=>x==null))Console.WriteLine("Exercice 3 à compléter");
else{if(repAffection.Any(x=>x==null))throw new Exception("Réponse partielle");for(int i=0;i<casAffection.Length;i++){var x=repAffection[i];var valeurs=casAffection[i];if(x.Affection==null||x.PartA==null||!x.Affection.SequenceEqual(valeurs)||x.PartA.Length!=valeurs.Length)throw new Exception("Dimensions incorrectes");for(int j=0;j<valeurs.Length;j++){var v=new Dictionary<string,double>(vA);v["affection"]=valeurs[j];double attendu=Partage(Faisceau(v,oA,wA).dep,db).partA;if(!double.IsFinite(x.PartA[j])||Math.Abs(x.PartA[j]-attendu)>1e-9)throw new Exception("Part incorrecte");}}var parts=repAffection[0].PartA;bool mono=Enumerable.Range(0,parts.Length-1).All(i=>parts[i]>parts[i+1]);if(!mono)throw new Exception("Monotonie incorrecte");Console.WriteLine($"Exercice 3 : sondes validées ; levier strictement décroissant : {mono}");}

Exercice 3 à compléter


## À retenir

- **Le désir n'est qu'une composante du faisceau de dépendance.** A ne perd en partant que `0.160` (affection), B en perd `0.340` (affection `0.060`, revenu `0.100`, logement `0.120`, statut `0.060`, réseau `0.000`) : l'écart `D_B − D_A = 0.180` est porté par la dépendance externe de B (`0.280` contre `0.000` pour A).
- **Un contre-exemple dans le modèle se retrouve dans les deux familles de poids informatives étudiées.** A désire **plus** (`desi_A = 0.160` contre `desi_B = 0.060`) et l'emporte quand même : `part_A = 0.746`, B ne capte que `0.254`. Sur **200** instances perturbées, le contre-exemple est reproduit **199/200** (99,5 %, arrondi à 100 % dans le tableau) avec un poids endogène (sigmoïde de l'écart), **200/200** (100 %) avec un poids exogène (ratio z), et **0/200** (0 %) avec le contrôle constant — qui donne `part_A = 0.500`.
- **La direction du levier est robuste ; son intensité ne l'est pas.** Le renversement tient pour tout `kappa > 0` dès que `D_B > D_A` : `part_A` va de `0.500` (kappa = 0.0) à `0.973` (kappa = 20.0), et de `0.595` (gamma = 0.5) à `0.998` (gamma = 8.0) dans la famille exogène. Le mécanisme est une propriété du **signe**, le chiffre une propriété de la **fonction**.
- **Les deux familles de poids ne voient pas les mêmes choses.** Quand le désir affectif de A monte de `0.5` à `1.0`, son levier **endogène** recule de `0.826` à `0.723`, tandis que son levier **exogène** reste à `0.823` : changer de famille, ce n'est pas changer un réglage, c'est changer ce que le poids voit.
- **Émanciper la dépendance externe réduit le levier brut et concentre l'asymétrie relationnelle.** En émancipant B composante par composante, `part_A` descend de `0.746` à `0.354` et la part de l'affection dans le faisceau de B monte de `17.6%` à `100.0%`.


Ces résultats sont conditionnels aux hypothèses des générateurs et ne réfutent pas, à eux seuls, une proposition empirique générale.

## Vérifiez votre compréhension

1. Une version simplifiée assimile l'intérêt au seul désir et annonce que la partie qui désire le moins contrôle le plus. Sur l'instance du carnet, A désire `0.160` contre `0.060` pour B, et A capte pourtant `0.746` du surplus. Où le principe échoue-t-il ?

   <details><summary>Réponse</summary>

   Le principe ne regarde qu'une composante. Le poids du marchandage dépend ici de l'écart de **dépendance** et non du désir : `D_B − D_A = 0.180`, et c'est la dépendance externe de B (`0.280`) qui fait basculer le levier. Le désir de A (`0.160` contre `0.060`) n'a même pas de canal dans la famille exogène — il y est constant.

   </details>

2. Le bras de contrôle « poids constant » donne `part_A = 0.500` et ne reproduit le contre-exemple que **0/200** fois. Qu'apporte ce bras qui n'apporte pas les deux autres ?

   <details><summary>Réponse</summary>

   Il **coupe tout canal** entre le faisceau et le poids : sans information sur le faisceau, le désir seul ne prédit rien — d'où la neutralité `0.500`. Ce bras montre seulement qu'un poids constant ne prédit aucune direction ; les conclusions restent conditionnelles aux générateurs étudiés. Il borne aussi ce qui peut être généralisé : seul le bras « endogène » porte l'effet partiel.

   </details>

3. Quand le désir affectif de A passe de `0.5` à `1.0`, le levier endogène recule de `0.826` à `0.723` mais le levier exogène reste à `0.823`. Peut-on alors conclure que « à faisceau externe égal, le désir fait basculer le levier » ?

   <details><summary>Réponse</summary>

   Non — seulement dans la famille endogène, où le poids voit le faisceau. Dans la famille exogène le désir n'a aucun canal : le tableau est le contre-exemple direct (le levier exogène ne bouge pas). La lecture précédente reste exacte **pour le modèle posé** ; la généraliser serait un artefact de famille.

   </details>

4. L'émancipation de B fait passer `part_A` de `0.746` à `0.354` et la part de l'affection dans le faisceau de B de `17.6%` à `100.0%`. Ces deux mouvements sont-ils le même phénomène ?

   <details><summary>Réponse</summary>

   Non, et le carnet le sépare explicitement. D'un côté le **levier brut** de A recule : la dépendance externe de B fond, l'écart de dépendance se retourne même au-delà de t = 0.8. De l'autre la **composition** de la dépendance de B se concentre : l'affection passe de 18 % à 100 % du faisceau. L'asymétrie restante est alors presque entièrement relationnelle — deux mouvements distincts, produits par la même bascule.

   </details>


## Pour aller plus loin

**Carnet suivant.** [GameTheory-04e — Oracles réflexifs](GameTheory-04e-Reflective-Oracles-Python.ipynb) -- l'autre forme de dépendance : non plus le faisceau social qui attache les joueurs l'un à l'autre, mais l'agent qui doit se demander ce qu'un autre agent, capable de le modéliser, ferait.

**Lectures.**

- Les références complètes du carnet sont dans la section « Notices historiques conservées du source » de la cellule « Limites et anti-claims » : Emerson (1962) sur les relations de pouvoir-dépendance, Nash (1950) sur le problème de la négociation, Rubinstein (1982) sur l'équilibre parfait en négociation, et le modèle d'investissement de Rusbult (1980) dont le faisceau est un parent pauvre.
- Le modèle exogène regarde les options extérieures, tandis que le modèle endogène regarde les écarts de dépendance. Cette différence est vérifiée par les calculs ; aucun résultat d’un modèle publié n’est reproduit par cette comparaison.

**Carnets voisins.**

- [GameTheory-04 — Nash](GameTheory-04-NashEquilibrium-Python.ipynb) -- le point de désaccord, dont ce carnet déplace la formulation.
- [GameTheory-04b — Lean NashExistence](GameTheory-04b-Lean-NashExistence-Lean.ipynb) -- l'existence formelle de l'équilibre, en Lean.
- [GameTheory-04c — NashExistence](GameTheory-04c-NashExistence-Python.ipynb) -- l'existence par point fixe de Brouwer, et le patron d'anti-tautologie (un vérificateur qui ne sait dire que « oui » ne teste rien) réutilisé ici.


**Retour** : [<< GT-04 Nash](GameTheory-04-NashEquilibrium-Python.ipynb) | [Index GameTheory](README.md)
